# IL / IR / US War Theater Map
### Interactive 3D OSINT Conflict Visualization

**Run the cell below** to launch the full interactive 3D globe. No npm, no build step needed.

| Feature | Description |
|---|---|
| 3D Globe | Night-Earth satellite texture, starfield, military-green atmosphere |
| LIVE Mode | Animates 3 random events simultaneously, cycles every 3.5s |
| TIMELINE Mode | Scrub Jan 2024 to Apr 2026, Play/Pause, 1x/5x/20x/100x speed |
| Animated Arcs | Color-coded: red=missile, blue=airstrike, orange=drone, cyan=naval |
| Event Cards | Click any arc or marker for full details panel on the right |
| Video | Click WATCH VIDEO FOOTAGE to search YouTube for event footage |
| News Ticker | Scrolling OSINT headlines in the top bar |
| 23 Events | Jan 2024 to Aug 2025 sourced, last 3 marked as projected scenarios |

**Controls:** Drag to rotate, scroll to zoom, click arc/dot to select event, Esc to close video

In [ ]:
import threading, http.server, socket, os, time
from IPython.display import IFrame, display

# Jupyter's HTML sanitizer strips <script> tags, so display(HTML(...)) breaks WebGL apps.
# Fix: spin up a tiny local HTTP server (daemon thread) so the file is served over
# a real http:// URL — the IFrame loads it as a proper page with full script support.

def _free_port():
    with socket.socket() as s:
        s.bind(('', 0))
        return s.getsockname()[1]

_serve_dir = os.path.abspath('.')
_port      = _free_port()

class _Handler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *a, **kw):
        super().__init__(*a, directory=_serve_dir, **kw)
    def log_message(self, *a):
        pass  # silence request logs

threading.Thread(
    target=lambda: http.server.HTTPServer(('127.0.0.1', _port), _Handler).serve_forever(),
    daemon=True   # dies automatically when the Jupyter kernel shuts down
).start()

time.sleep(0.4)  # let the server bind before the iframe requests the file

url = f'http://127.0.0.1:{_port}/war_map.html'
print(f'Serving from: {_serve_dir}')
print(f'Globe URL   : {url}')
display(IFrame(url, width='100%', height=880))

---
## OSINT Events Dataset
All 23 events plotted on the globe. Run the cell below to see the full structured dataset.

In [ ]:
events = [
    {"id":1,  "date":"2024-01-12", "title":"Operation Prosperity Guardian",          "type":"airstrike", "actor":"USA",       "importance":"major",    "sim":False},
    {"id":2,  "date":"2024-01-28", "title":"Tower 22 Drone Strike",                  "type":"drone",     "actor":"Iraq PMF",  "importance":"critical", "sim":False},
    {"id":3,  "date":"2024-02-02", "title":"Op Inherent Resolve Retaliation",        "type":"airstrike", "actor":"USA",       "importance":"critical", "sim":False},
    {"id":4,  "date":"2024-04-01", "title":"Strike on Iranian Consulate Damascus",   "type":"airstrike", "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":5,  "date":"2024-04-13", "title":"Operation True Promise",                "type":"missile",   "actor":"Iran",      "importance":"critical", "sim":False},
    {"id":6,  "date":"2024-04-14", "title":"Houthi Coordinated Salvo",              "type":"missile",   "actor":"Houthi",    "importance":"major",    "sim":False},
    {"id":7,  "date":"2024-04-19", "title":"Israeli Strike - Isfahan",              "type":"drone",     "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":8,  "date":"2024-07-27", "title":"Majdal Shams Massacre",                "type":"missile",   "actor":"Hezbollah", "importance":"critical", "sim":False},
    {"id":9,  "date":"2024-09-17", "title":"Pager Operation",                      "type":"ground",    "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":10, "date":"2024-09-27", "title":"Nasrallah Eliminated",                 "type":"airstrike", "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":11, "date":"2024-10-01", "title":"IDF Ground Invasion - Lebanon",         "type":"ground",    "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":12, "date":"2024-10-01", "title":"Operation True Promise II",            "type":"missile",   "actor":"Iran",      "importance":"critical", "sim":False},
    {"id":13, "date":"2024-10-15", "title":"USS Abraham Lincoln Enters Gulf",       "type":"naval",     "actor":"USA",       "importance":"major",    "sim":False},
    {"id":14, "date":"2024-10-26", "title":"Operation Days of Repentance",         "type":"airstrike", "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":15, "date":"2024-11-26", "title":"Lebanon Ceasefire",                    "type":"defense",   "actor":"USA",       "importance":"major",    "sim":False},
    {"id":16, "date":"2025-01-19", "title":"Gaza Ceasefire - Phase 1",             "type":"defense",   "actor":"USA",       "importance":"critical", "sim":False},
    {"id":17, "date":"2025-03-18", "title":"Gaza Ceasefire Collapses",             "type":"airstrike", "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":18, "date":"2025-04-12", "title":"US-Iran Nuclear Talks - Oman",         "type":"defense",   "actor":"USA",       "importance":"major",    "sim":False},
    {"id":19, "date":"2025-05-05", "title":"US Strikes Houthi Capital",            "type":"airstrike", "actor":"USA",       "importance":"critical", "sim":False},
    {"id":20, "date":"2025-08-20", "title":"Israeli Strike - Natanz Facility",     "type":"airstrike", "actor":"Israel",    "importance":"critical", "sim":False},
    {"id":21, "date":"2026-01-15", "title":"Operation True Promise III [PROJ]",    "type":"missile",   "actor":"Iran",      "importance":"critical", "sim":True},
    {"id":22, "date":"2026-03-08", "title":"US & Israel Joint Strike IRGC [PROJ]", "type":"airstrike", "actor":"USA",       "importance":"critical", "sim":True},
    {"id":23, "date":"2026-04-10", "title":"USS Carl Vinson - Hormuz [PROJ]",      "type":"naval",     "actor":"USA",       "importance":"critical", "sim":True},
]

from collections import Counter
print(f'Total events : {len(events)}')
print(f'Date range   : {events[0]["date"]} to {events[-1]["date"]}')
print(f'Confirmed    : {sum(1 for e in events if not e["sim"])} / Projected: {sum(1 for e in events if e["sim"])}')
print(f'Critical     : {sum(1 for e in events if e["importance"]=="critical")}')
print()
print('By type :', dict(Counter(e['type'] for e in events)))
print('By actor:', dict(Counter(e['actor'] for e in events)))
print()
print(f'{"ID":<4} {"DATE":<12} {"ACTOR":<12} {"TYPE":<10} {"IMP":<10} {"SIM":<5} TITLE')
print('-'*90)
for e in events:
    print(f'{e["id"]:<4} {e["date"]:<12} {e["actor"]:<12} {e["type"]:<10} {e["importance"]:<10} {"[P]" if e["sim"] else "   "}   {e["title"]}')

---
## News Headlines Feed
These scroll in the ticker at the top of the globe. The React version can connect to NewsAPI.org for live headlines via `VITE_NEWS_API_KEY` in `.env`.

In [ ]:
headlines = [
    ("Reuters",        "Israel signals readiness for further strikes if nuclear talks fail"),
    ("AP",             "Iran enriches uranium to 84% purity - IAEA gravely concerned"),
    ("Defense News",   "US deploys additional THAAD battery to Israel amid Iran tensions"),
    ("Lloyd's List",   "Houthi attack damages Singapore-flagged tanker in Red Sea"),
    ("Times of Israel","Netanyahu: military option against Iran remains on the table"),
    ("Al Jazeera",     "Iran supreme leader warns of consequences of further Israeli strikes"),
    ("WSJ",            "US-Iran Oman talks stall over enrichment caps"),
    ("IDF",            "300+ rockets fired from Gaza since ceasefire collapsed"),
    ("NYT",            "Russia reportedly shares satellite intelligence with Iran"),
    ("Bloomberg",      "Saudi Arabia warns regional war could spike oil to $200/barrel"),
    ("Tasnim",         "IRGC test-fires Fatah-2 hypersonic in Persian Gulf exercise"),
    ("BBC",            "UK joins US in Red Sea escort operations amid record Houthi attacks"),
    ("Haaretz",        "Israel conducts largest-ever exercise simulating Iran strikes"),
    ("Reuters",        "Hezbollah fires 80 rockets toward Galilee - IDF intercepts majority"),
    ("FT",             "Oil surges 4% on potential Strait of Hormuz closure fears"),
    ("Pentagon",       "USS Carl Vinson combat-ready in Persian Gulf"),
    ("IRNA",           "Iran announces new missile with 2,000km range"),
    ("Axios",          "Israel preparing 'day after' plan for Iran nuclear program"),
]

print(f'{"SOURCE":<18} HEADLINE')
print('-'*80)
for src, line in headlines:
    print(f'[{src}]{" "*(16-len(src))} {line}')

---
## Project Structure

```
stam/
  war_map.html          <- Standalone globe (CDN, no build needed)
  stan.ipynb            <- This notebook
  package.json          <- React/Vite version
  src/
    App.jsx             <- Main layout & state
    components/
      Globe3D.jsx       <- react-globe.gl wrapper
      Timeline.jsx      <- Scrubber + play controls
      EventCard.jsx     <- Right-panel event details
      VideoModal.jsx    <- YouTube embed modal
      NewsFeed.jsx      <- Scrolling news ticker
      Header.jsx        <- Top bar + mode toggle
      Legend.jsx        <- Arc type legend
    data/
      events.js         <- 23 OSINT events
      newsHeadlines.js  <- Fallback headlines
    hooks/
      useSimulation.js  <- Live/timeline animation
      useNews.js        <- NewsAPI fetching
    utils/colors.js
```

**React version (full features):**
```bash
npm install && npm run dev
```

**Add live news headlines:**
```bash
echo 'VITE_NEWS_API_KEY=your_key' > .env
```

---
## YouTube Video ID Scraper
Run this cell once to scrape real YouTube video IDs for each event.
Results are saved to `video_ids.json` and loaded automatically by the globe.
After running, **re-run the globe cell above** to reload the map — embedded video will now work inside the app.

In [ ]:
import requests, re, json, time

QUERIES = {
    1:  'Tower 22 drone attack Jordan US soldiers killed 2024',
    2:  'US airstrike Iraq Syria retaliation Tower 22 February 2024',
    3:  'Israel strike Iranian consulate Damascus April 2024 IRGC generals',
    4:  'Operation True Promise Iran missile attack Israel April 13 2024',
    5:  'Houthi missiles Israel April 14 2024 coordinated attack',
    6:  'Israel drone strike Isfahan Iran April 19 2024',
    7:  'Majdal Shams Hezbollah rocket attack children killed July 2024',
    8:  'US UK strikes Yemen Houthi January 12 2024 Operation Prosperity Guardian',
    9:  'Lebanon pager explosion Hezbollah September 17 2024',
    10: 'Israel kills Hassan Nasrallah Hezbollah Beirut September 2024',
    11: 'IDF ground invasion Lebanon October 2024 Hezbollah',
    12: 'Operation True Promise 2 Iran missile attack Israel October 2024',
    13: 'USS Abraham Lincoln Persian Gulf three carrier deployment Middle East 2024',
    14: 'Israel strikes Iran October 26 2024 missile production air defense',
    15: 'Lebanon ceasefire November 2024 Israel Hezbollah US brokered',
    16: 'Gaza ceasefire January 2025 hostages Hamas Israel Qatar',
    17: 'Israel resumes Gaza operation March 2025 ceasefire collapse',
    18: 'US Iran nuclear talks Oman 2025 Witkoff Araghchi',
    19: 'US B-2 bomber Houthi Sanaa strike 2025 bunker buster',
    20: 'Israel strikes Natanz nuclear facility Iran 2025',
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

video_ids = {}
for ev_id, query in QUERIES.items():
    try:
        url = 'https://www.youtube.com/results?search_query=' + requests.utils.quote(query)
        r = requests.get(url, headers=HEADERS, timeout=12)
        ids = re.findall(r'"videoId":"([a-zA-Z0-9_\-]{11})"', r.text)
        # Filter out channel/playlist IDs (appear in 'browseId' context)
        if ids:
            video_ids[str(ev_id)] = ids[0]
            print(f'  Event {ev_id:2d}: {ids[0]}  — {query[:55]}')
        else:
            print(f'  Event {ev_id:2d}: no results for "{query[:50]}"')
        time.sleep(1.2)  # polite delay
    except Exception as e:
        print(f'  Event {ev_id:2d}: ERROR — {e}')

with open('video_ids.json', 'w') as f:
    json.dump(video_ids, f, indent=2)

print(f'\nSaved {len(video_ids)}/{len(QUERIES)} IDs to video_ids.json')
print('Re-run the globe cell above to enable embedded video playback.')


---
## מדריך Git — שלושת הפקודות הבסיסיות

כל פרויקט מתחיל בשלושה שלבים:

### 1. `git add .`
מוסיף את **כל הקבצים שהשתנו** ל-"staging area" — הרשימה של מה שיכנס ל-commit.
הנקודה (`.`) אומרת: "הכל בתיקייה הנוכחית".

```bash
git add .          # הוסף הכל
git add index.html # או רק קובץ ספציפי
```

### 2. `git commit -m "תיאור"`
**שומר snapshot** של השינויים עם הודעה שמסבירה מה עשית.
ה-commit הוא כמו "שמירה" בתוך ההיסטוריה של הפרויקט — אפשר תמיד לחזור אליו.

```bash
git commit -m "add war map initial version"
```

> ✅ כלל אצבע: commit אחד = שינוי אחד ברור. לא 10 דברים ביחד.

### 3. `git push origin main`
**מעלה את ה-commits** מהמחשב שלך ל-GitHub.
- `origin` = השם של ה-remote (GitHub)
- `main` = שם ה-branch

```bash
git push origin main
```

### 4. `git pull`
**מוריד שינויים** מ-GitHub למחשב שלך (כשמישהו אחר דחף קוד, או כשעבדת ממחשב אחר).

```bash
git pull           # מוריד ומיזג אוטומטית
```

---

### זרימת עבודה יומיומית

```
┌─────────────────────────────────────────────────────┐
│  שינית קבצים?                                       │
│                                                     │
│   git add .                                         │
│   git commit -m "מה שינית ולמה"                    │
│   git push origin main                              │
│                                                     │
│  חוזר על עצמו בכל פעם שרוצים לשמור את העבודה.     │
└─────────────────────────────────────────────────────┘
```

| פקודה | מה היא עושה |
|---|---|
| `git add .` | מסמן מה נכנס ל-commit |
| `git commit -m "..."` | שומר snapshot עם הודעה |
| `git push origin main` | מעלה ל-GitHub |
| `git pull` | מוריד שינויים מ-GitHub |